# CNN-Based Classification of Images Containing One Person or Multiple People
**Course work – Neural Networks | Topic: Comparison of convolutional neural network models for image classification**

**Aim:** decide whether a photograph shows a **single person** or a **group of people**, and compare three CNN models.

## Workflow of this notebook
| Step | What we do | Course-work chapter |
|---|---|---|
| 0 | Set up the tools and settings | – |
| 1 | Build the labelled dataset and split it into train / validation / test | Dataset |
| 2 | Look at the data (class balance, image sizes, examples) | Dataset analysis |
| 3 | Preprocessing and data augmentation | Preprocessing |
| 4 | Define the models: **custom CNN, ResNet50, MobileNetV2** | Models |
| 5 | Train 6 experiments (transfer learning and augmentation are tested here) | Experiments |
| 6 | Evaluate: accuracy, precision, recall, F1, confusion matrices | Results |
| 7 | Effect of transfer learning and of augmentation | Discussion |
| 8 | Computational efficiency (size, training time, speed) | Discussion |
| 9 | Look at misclassified images and find the reasons | Error analysis |
| 10 | **Try your own photo: choose a file → SINGLE or GROUP + number of people** | Demonstration |
| 11 | Conclusions | Conclusion |

**How to run:** *Runtime → Change runtime type → GPU (T4)*, then *Runtime → Run all* (about 30–40 min).
To test quickly first, set `QUICK_TEST = True` in Step 0, run all, then set it back to `False`.

## Step 0 – Setup and settings

In [ ]:
import os, json, time, zipfile, random, gc, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests, urllib3
from PIL import Image, ImageOps
from IPython.display import display

import tensorflow as tf

# Let TensorFlow take GPU memory only when needed, so PyTorch (person detector in Step 10) can also use the GPU
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU") or "none (switch runtime to GPU!)")

In [8]:
# ---------------- SETTINGS (you can change these) ----------------
QUICK_TEST      = False    # True = tiny dataset and 2 epochs, only to check that the notebook runs
IMG_SIZE        = 160      # every image is resized to IMG_SIZE x IMG_SIZE
BATCH           = 32
EPOCHS          = 10
SEED            = 42

N_PER_CLASS     = 1500     # images per class  (balanced dataset)
GROUP_MIN       = 3        # "group" = at least 3 clearly visible people
MIN_PERSON_FRAC = 0.02     # a person is "clearly visible" if its box covers >= 2 % of the image
COCO_SPLIT      = "train2017"   # use "val2017" if Colab runs out of memory

if QUICK_TEST:
    N_PER_CLASS, EPOCHS, COCO_SPLIT = 120, 2, "val2017"

CLASS_NAMES = ["single", "group"]      # label 0 = single, label 1 = group
WORK = Path(os.environ.get("WORK_DIR", "/content/work"))
(WORK / "images").mkdir(parents=True, exist_ok=True)
(WORK / "weights").mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

## Step 1 – Dataset
We use real photographs from the public **MS COCO** dataset. Every photo has the people marked with boxes, so the label is created automatically:

| Class | Rule |
|---|---|
| **single** (0) | exactly **one** person, clearly visible |
| **group** (1) | **3 or more** clearly visible people |
| *not used* | images without people, with exactly 2 people (unclear case), crowds, or only tiny people |

The number of people is also saved (`n_people`). It is **not** used for training, but it helps in the error analysis (Step 9).

*Note:* the COCO server `images.cocodataset.org` does **not** have a valid HTTPS certificate, so files are downloaded over plain **http** (this was the cause of the `SSLCertVerificationError`).

In [9]:
# Helper: download a file from COCO. COCO only works correctly over http:// (its https certificate is invalid).
def coco_url(url):
    return url.replace("https://", "http://")

def download_file(url, path, timeout=120, retries=4):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return True
    tmp = path.with_suffix(path.suffix + ".part")
    for attempt in range(retries):
        for u in (coco_url(url), url.replace("http://", "https://")):      # 1st try http, then https without cert check
            try:
                with requests.get(u, stream=True, timeout=timeout, verify=False) as r:
                    r.raise_for_status()
                    with open(tmp, "wb") as f:
                        for chunk in r.iter_content(1 << 20):
                            f.write(chunk)
                tmp.rename(path)          # rename only after the whole file arrived (no half-downloaded files)
                return True
            except Exception as e:
                last_error = e
                tmp.unlink(missing_ok=True)
        time.sleep(1 + attempt)
    print("Download failed:", url, "|", last_error)
    return False

In [ ]:
# 1a) Read the COCO annotations and decide the label of every image
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"   # ~250 MB
ann_zip = WORK / "annotations.zip"
print("Downloading COCO annotations (only the first time, ~250 MB) ...")
assert download_file(ANN_URL, ann_zip, timeout=300), "Could not download the annotations - run the cell again."

try:
    with zipfile.ZipFile(ann_zip) as z:
        with z.open(f"annotations/instances_{COCO_SPLIT}.json") as fh:
            coco = json.load(fh)
except zipfile.BadZipFile:
    ann_zip.unlink(missing_ok=True)
    raise RuntimeError("The annotation zip was damaged and has been deleted - please run this cell again.")

images_info = {im["id"]: im for im in coco["images"]}

people = {}                                    # image id -> list of person annotations
for a in coco["annotations"]:
    if a["category_id"] == 1:                  # category 1 = person
        people.setdefault(a["image_id"], []).append(a)

rows = []
for image_id, anns in people.items():
    im = images_info[image_id]
    if any(a["iscrowd"] for a in anns):
        continue
    sizes = [a["bbox"][2] * a["bbox"][3] / (im["width"] * im["height"]) for a in anns]   # person size / image size
    big = [s for s in sizes if s >= MIN_PERSON_FRAC]
    if len(sizes) == 1 and len(big) == 1:
        label = "single"
    elif len(big) >= GROUP_MIN:
        label = "group"
    else:
        continue
    rows.append(dict(image_id=image_id, url=coco_url(im["coco_url"]),
                     width=im["width"], height=im["height"], label=label,
                     n_people=len(big), person_size=float(np.mean(big))))
del coco, images_info, people; gc.collect()
candidates = pd.DataFrame(rows)
print(candidates.label.value_counts().to_string())

In [ ]:
# 1b) Pick a balanced set of images and download them
n = min(N_PER_CLASS, int(candidates.label.value_counts().min()))
df = pd.concat([candidates[candidates.label == c].sample(n, random_state=SEED) for c in CLASS_NAMES], ignore_index=True)
for c in CLASS_NAMES:
    (WORK / "images" / c).mkdir(exist_ok=True)
df["path"] = [str(WORK / "images" / r.label / f"{r.image_id}.jpg") for r in df.itertuples()]

def download_image(url, path):
    if not download_file(url, path, timeout=30, retries=3):
        return False
    try:
        with Image.open(path) as im:
            im.convert("RGB")                 # check that the file is a real image
        return True
    except Exception:
        Path(path).unlink(missing_ok=True)
        return False

print(f"Downloading {len(df)} images ...")
start = time.time()
with ThreadPoolExecutor(16) as pool:
    df["ok"] = list(pool.map(download_image, df.url, df.path))
print(f"Failed downloads: {(~df.ok).sum()}  |  time: {time.time() - start:.0f} s")
df = df[df.ok].drop(columns="ok").reset_index(drop=True)
df["y"] = df.label.map({"single": 0, "group": 1})
assert len(df) > 50, "Almost no images were downloaded - check the internet connection and run the cell again."
print("Images ready:", len(df))
print(df.label.value_counts().to_string())

In [ ]:
# 1c) Split: 70 % train, 15 % validation, 15 % test (same class balance in each part)
train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df.y, random_state=SEED)
val_df, test_df   = train_test_split(rest_df, test_size=0.50, stratify=rest_df.y, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

split_table = pd.DataFrame({"train": train_df.label.value_counts(), "validation": val_df.label.value_counts(),
                            "test": test_df.label.value_counts()}).T[CLASS_NAMES]
split_table["total"] = split_table.sum(axis=1)
print(split_table)

## Step 2 – Look at the data
Class balance, image resolutions, people per image and some examples. Write your comments about these plots in the report.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="label", order=CLASS_NAMES, ax=ax[0]);                ax[0].set_title("Images per class")
ax[1].scatter(df.width, df.height, s=6, alpha=.4);                             ax[1].set_title("Original image sizes")
ax[1].set_xlabel("width (px)"); ax[1].set_ylabel("height (px)")
sns.histplot(df[df.label == "group"].n_people, discrete=True, ax=ax[2], color="tab:orange")
ax[2].set_title("People per image (group class)"); ax[2].set_xlabel("number of people")
plt.tight_layout(); plt.show()
print(df.groupby("label")[["n_people", "person_size"]].mean().round(3))

In [ ]:
# Example images (number of people in the title)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, cls in enumerate(CLASS_NAMES):
    subset = df[df.label == cls]
    for col, r in enumerate(subset.sample(min(6, len(subset)), random_state=1).itertuples()):
        axes[row, col].imshow(Image.open(r.path).convert("RGB"))
        axes[row, col].set_title(f"{cls} ({r.n_people})")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Step 3 – Preprocessing and augmentation
* **Resize** every image to 160 × 160 pixels (all networks need the same input size).
* **Normalise** the pixel values – done inside each model, because each network expects a different range.
* **Augmentation** (only during training, optional in the model): random horizontal flip, small rotation, zoom and contrast change. It creates "new" training images and reduces over-fitting.

In [ ]:
def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(tf.clip_by_value(img, 0, 255)), tf.uint8), label

def make_dataset(frame, training):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values.astype(str), frame.y.values.astype("float32")))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).cache()      # decode every image only once
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y))
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_dataset(train_df, True), make_dataset(val_df, False), make_dataset(test_df, False)

def make_augmentation():
    # a NEW augmentation block for every model (sharing one block between models causes errors in Keras 3)
    return keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.08),
                             layers.RandomZoom(0.15), layers.RandomContrast(0.2)], name="augmentation")

# show the effect of augmentation on one image
augmentation_demo = make_augmentation()
sample = np.array(Image.open(train_df.path[0]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(sample); axes[0].set_title("resized original")
for i in range(1, 5):
    aug = augmentation_demo(sample[None].astype("float32"), training=True)[0]
    axes[i].imshow(np.clip(np.asarray(aug), 0, 255).astype("uint8"))
    axes[i].set_title(f"augmented {i}")
for a in axes: a.axis("off")
plt.show()

## Step 4 – The three CNN models
| Model | Idea | Parameters |
|---|---|---|
| **Custom CNN** | our own small network: 4 blocks of (Conv → BatchNorm → ReLU) ×2 + MaxPooling | ~1.2 M |
| **ResNet50** | 50 layers with *residual (skip) connections*, very accurate | ~24 M |
| **MobileNetV2** | *depthwise-separable convolutions*, small and fast | ~2.3 M |

ResNet50 and MobileNetV2 can start from **random weights** ("scratch") or from **ImageNet weights** ("pretrained" = transfer learning; here all layers are re-trained with a small learning rate, BatchNorm statistics stay frozen).
All models get the same classification end: *global average pooling → dropout → 1 neuron with sigmoid* (output = probability of "group").

In [ ]:
class ResNetPreprocess(layers.Layer):
    # Same as keras.applications.resnet50.preprocess_input ("caffe" mode): RGB -> BGR and subtract the ImageNet mean.
    # Written as a layer so that saving / loading weights works without Lambda-layer problems.
    def call(self, x):
        x = x[..., ::-1]
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)

def build_model(arch, pretrained=False, augment=True):
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = make_augmentation()(inputs) if augment else inputs

    if arch == "custom":
        x = layers.Rescaling(1 / 255)(x)                      # pixels 0..255 -> 0..1
        for filters in (32, 64, 128, 256):
            for _ in range(2):
                x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
                x = layers.BatchNormalization()(x)
                x = layers.ReLU()(x)
            x = layers.MaxPooling2D()(x)
    else:
        weights = "imagenet" if pretrained else None
        if arch == "resnet50":
            base = keras.applications.ResNet50(include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
            x = ResNetPreprocess()(x)
        elif arch == "mobilenetv2":
            base = keras.applications.MobileNetV2(include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
            x = layers.Rescaling(1 / 127.5, offset=-1)(x)     # pixels -> -1..1
        else:
            raise ValueError(f"unknown architecture {arch}")
        x = base(x, training=False) if pretrained else base(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name=f"{arch}_{'pre' if pretrained else 'scratch'}_{'aug' if augment else 'noaug'}")

build_model("custom").summary()

## Step 5 – Experiments
Six experiments answer the research questions:

| Experiment | Question it answers |
|---|---|
| CustomCNN_noAug vs. CustomCNN_Aug | Does **data augmentation** help? |
| ResNet50_scratch vs. ResNet50_pretrained | Does **transfer learning** help? |
| MobileNetV2_scratch vs. MobileNetV2_pretrained | Does **transfer learning** help? |
| all models together | Which CNN is best, and at what computational cost? |

Training stops early if the validation loss does not improve for 4 epochs; the best epoch is kept.

In [ ]:
EXPERIMENTS = {   # name: (architecture, pretrained, augmentation)
    "CustomCNN_noAug":        ("custom",      False, False),
    "CustomCNN_Aug":          ("custom",      False, True),
    "ResNet50_scratch":       ("resnet50",    False, True),
    "ResNet50_pretrained":    ("resnet50",    True,  True),
    "MobileNetV2_scratch":    ("mobilenetv2", False, True),
    "MobileNetV2_pretrained": ("mobilenetv2", True,  True),
}
RESULTS = {}

def train_and_test(name, arch, pretrained, augment):
    keras.backend.clear_session(); gc.collect(); keras.utils.set_random_seed(SEED)
    model = build_model(arch, pretrained, augment)
    lr = 1e-4 if pretrained else 1e-3            # small learning rate when starting from ImageNet weights
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
                 keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)]
    print(f"\n=== {name} ({model.count_params():,} parameters) ===")

    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
    train_time = time.time() - start

    model.predict(test_ds.take(1), verbose=0)                       # warm-up
    start = time.time()
    prob = model.predict(test_ds, verbose=0).ravel()                # probability of "group" for every test image
    infer_ms = 1000 * (time.time() - start) / len(test_df)

    model.save_weights(str(WORK / "weights" / f"{name}.weights.h5"))
    RESULTS[name] = dict(history={k: [float(v) for v in vals] for k, vals in history.history.items()},
                         prob=prob, params=model.count_params(),
                         train_min=train_time / 60, infer_ms=infer_ms, epochs=len(history.history["loss"]),
                         config=(arch, pretrained, augment))
    print(f"-> test accuracy {accuracy_score(test_df.y, prob >= .5):.3f}  |  training {train_time / 60:.1f} min")
    del model; gc.collect()

for name, (arch, pretrained, augment) in EXPERIMENTS.items():
    if name in RESULTS:            # already trained (useful if you re-run the cell after a disconnect)
        continue
    train_and_test(name, arch, pretrained, augment)

In [ ]:
# Training curves: accuracy and loss for every experiment (train vs. validation)
fig, axes = plt.subplots(len(RESULTS), 2, figsize=(11, 3 * len(RESULTS)), squeeze=False)
for row, (name, r) in enumerate(RESULTS.items()):
    h = r["history"]
    axes[row, 0].plot(h["accuracy"], label="train"); axes[row, 0].plot(h["val_accuracy"], label="validation")
    axes[row, 0].set_title(f"{name} - accuracy"); axes[row, 0].set_xlabel("epoch"); axes[row, 0].legend()
    axes[row, 1].plot(h["loss"], label="train"); axes[row, 1].plot(h["val_loss"], label="validation")
    axes[row, 1].set_title(f"{name} - loss"); axes[row, 1].set_xlabel("epoch"); axes[row, 1].legend()
plt.tight_layout(); plt.show()

## Step 6 – Evaluation on the test set
The test images were never used for training or for choosing the best epoch. "Group" is the positive class.

* **Accuracy** – share of correct answers
* **Precision** – of the images called *group*, how many really are groups
* **Recall** – of all real groups, how many were found
* **F1-score** – balance of precision and recall
* **Confusion matrix** – how many images of each true class were predicted as each class

In [ ]:
y_true = test_df.y.values.astype(int)
rows = []
for name, r in RESULTS.items():
    pred = (r["prob"] >= 0.5).astype(int)
    rows.append(dict(model=name, accuracy=accuracy_score(y_true, pred),
                     precision=precision_score(y_true, pred, zero_division=0),
                     recall=recall_score(y_true, pred, zero_division=0),
                     f1=f1_score(y_true, pred, zero_division=0)))
metrics = pd.DataFrame(rows).set_index("model").round(3)
display(metrics.style.background_gradient(cmap="Greens").format("{:.3f}"))
metrics.to_csv(WORK / "test_metrics.csv")       # table for the report
best = metrics.f1.idxmax()
print("Best model (highest F1):", best)

In [ ]:
# Confusion matrices of all models
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax in axes.ravel(): ax.axis("off")
for ax, (name, r) in zip(axes.ravel(), RESULTS.items()):
    ax.axis("on")
    cm = confusion_matrix(y_true, (r["prob"] >= .5).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(name); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

## Step 7 – Effect of transfer learning and data augmentation
Difference in F1-score (positive number = the first option is better).

In [ ]:
f1 = metrics.f1
effects = pd.DataFrame({
    "comparison": ["Augmentation (custom CNN): with - without",
                   "Transfer learning (ResNet50): pretrained - scratch",
                   "Transfer learning (MobileNetV2): pretrained - scratch"],
    "F1 difference": [f1["CustomCNN_Aug"] - f1["CustomCNN_noAug"],
                      f1["ResNet50_pretrained"] - f1["ResNet50_scratch"],
                      f1["MobileNetV2_pretrained"] - f1["MobileNetV2_scratch"]]}).set_index("comparison").round(3)
display(effects)

# gap between train and validation accuracy in the last epoch (bigger gap = more over-fitting)
gap = pd.Series({n: r["history"]["accuracy"][-1] - r["history"]["val_accuracy"][-1] for n, r in RESULTS.items()},
                name="train - val accuracy (last epoch)").round(3)
display(gap.to_frame())

## Step 8 – Computational efficiency
Parameters, training time and prediction speed per image (speed depends on the GPU, but the *differences between models* are what matters).

In [ ]:
efficiency = pd.DataFrame({n: dict(parameters_millions=r["params"] / 1e6, epochs_trained=r["epochs"],
                                   training_minutes=r["train_min"], ms_per_image=r["infer_ms"], f1=metrics.f1[n])
                           for n, r in RESULTS.items()}).T.astype(float).round(3)
display(efficiency)
efficiency.to_csv(WORK / "efficiency.csv")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(efficiency.ms_per_image, efficiency.f1, s=40 + 15 * efficiency.parameters_millions, alpha=.7)
for n in efficiency.index:
    ax.annotate(n, (efficiency.ms_per_image[n], efficiency.f1[n]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("time per image (ms)"); ax.set_ylabel("test F1"); ax.set_title("Accuracy vs. speed (bubble size = parameters)")
plt.show()

## Step 9 – Misclassified images
We look at the mistakes of the best model and check whether they depend on the number of people.

In [ ]:
errors = test_df.copy()
errors["prob"] = RESULTS[best]["prob"]
errors["pred"] = (errors.prob >= .5).astype(int)
errors["wrong"] = errors.pred != errors.y
print(f"{best}: {errors.wrong.sum()} wrong of {len(errors)} test images")

wrong_images = (errors[errors.wrong].assign(confidence=lambda d: (d.prob - .5).abs())
                .sort_values("confidence", ascending=False).head(8))
if len(wrong_images) == 0:
    print("No mistakes on the test set!")
else:
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for ax in axes.ravel(): ax.axis("off")
    for ax, r in zip(axes.ravel(), wrong_images.itertuples()):
        ax.imshow(Image.open(r.path).convert("RGB"))
        ax.set_title(f"true: {CLASS_NAMES[r.y]} | predicted: {CLASS_NAMES[r.pred]}\n{r.n_people} people | P(group)={r.prob:.2f}", fontsize=9)
    plt.suptitle("Most confident mistakes"); plt.tight_layout(); plt.show()

In [ ]:
# Do the mistakes depend on how many people are in the picture, or how big they are?
errors["people"] = pd.cut(errors.n_people, [0, 1, 4, 7, 1000], labels=["1", "3-4", "5-7", "8+"])
# rank() avoids the "bin edges must be unique" error of qcut
errors["person size"] = pd.qcut(errors.person_size.rank(method="first"), 3, labels=["small", "medium", "large"])
print("Error rate (%) by number of people:")
print((errors.groupby("people", observed=True).wrong.agg(["mean", "size"])
       .rename(columns={"mean": "error %", "size": "images"}).assign(**{"error %": lambda d: (d["error %"] * 100).round(1)})).to_string())
print("\nError rate (%) by size of the people:")
print((errors.groupby("person size", observed=True).wrong.agg(["mean", "size"])
       .rename(columns={"mean": "error %", "size": "images"}).assign(**{"error %": lambda d: (d["error %"] * 100).round(1)})).to_string())

**Possible reasons for errors (check them on your pictures and write the examples in the report):**
small or distant people · people hidden or cut by the image border · one big person in front with others far behind · statues, posters, screens or reflections that look like people · label noise (tiny extra people that were not counted) · the model learns the scene ("street", "party") instead of counting people · low resolution (160 px) makes background people disappear.

## Step 10 – Try your own photo
Run the first cell (loads the models), then the second cell and click **Choose files**. The best CNN from Step 6 decides **single or group**; a pretrained person detector additionally counts the people (the counting is only a bonus – the course-work task is the classification).

Use **JPG or PNG** photos (iPhone *HEIC* photos are not supported – convert them to JPG first). Phone photo rotation (EXIF) is corrected automatically.

In [ ]:
# Load the best classifier again (from the saved weights) and a pretrained person detector
import torch, torchvision
from torchvision.ops import nms, box_area
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

DET_SCORE   = 0.5    # only boxes the detector is quite sure about
DET_NMS_IOU = 0.7    # boxes that overlap a lot are the same person
DET_INSIDE  = 0.95   # a box lying almost completely inside a bigger one is a part of that person (duplicate)

arch, pretrained, augment = RESULTS[best]["config"]
keras.backend.clear_session()
classifier = build_model(arch, pretrained, augment)
classifier.load_weights(str(WORK / "weights" / f"{best}.weights.h5"))
print("Classifier:", best)

device = "cuda" if torch.cuda.is_available() else "cpu"
# ResNet50-FPN v2 detector: slower than the MobileNet one, but it counts people in close groups much more reliably
detector = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT).to(device).eval()
print("Person detector ready on", device)

def count_people(img):
    tensor = torchvision.transforms.functional.to_tensor(img).to(device)
    with torch.no_grad():
        out = detector([tensor])[0]
    keep = (out["labels"] == 1) & (out["scores"] >= DET_SCORE)
    boxes, scores = out["boxes"][keep], out["scores"][keep]
    # 1) remove strongly overlapping duplicates
    k = nms(boxes, scores, DET_NMS_IOU)
    boxes, scores = boxes[k], scores[k]
    # 2) remove boxes that are only a part of another person (e.g. face/upper body inside a full-body box)
    areas = box_area(boxes)
    final = []
    for i in torch.argsort(areas, descending=True).tolist():          # big boxes first
        duplicate = False
        for j in final:
            x1 = torch.max(boxes[i, 0], boxes[j, 0]); y1 = torch.max(boxes[i, 1], boxes[j, 1])
            x2 = torch.min(boxes[i, 2], boxes[j, 2]); y2 = torch.min(boxes[i, 3], boxes[j, 3])
            inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
            if inter / areas[i] > DET_INSIDE:
                duplicate = True; break
        if not duplicate:
            final.append(i)
    boxes = boxes[final].cpu().numpy()
    # 3) keep only clearly visible people (same rule as in the dataset)
    return [b for b in boxes if (b[2] - b[0]) * (b[3] - b[1]) / (img.width * img.height) >= MIN_PERSON_FRAC]

def predict_photo(path):
    try:
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")      # fix rotation of phone photos
    except Exception as e:
        print(f"Cannot open {path}: {e}  (use JPG or PNG)")
        return

    # 1) classification: our CNN (same preprocessing as in training: resize to 160x160, values 0..255)
    x = tf.round(tf.image.resize(np.asarray(img, dtype=np.float32), (IMG_SIZE, IMG_SIZE)))[None]
    p_group = float(classifier.predict(x, verbose=0)[0, 0])
    answer = CLASS_NAMES[int(p_group >= 0.5)].upper()

    # 2) counting: pretrained detector (only people that are clearly visible)
    boxes = count_people(img)

    plt.figure(figsize=(8, 7)); plt.imshow(img); plt.axis("off")
    for b in boxes:
        plt.gca().add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2))
    plt.title(f"{answer}  (P(group) = {p_group:.2f})   |   people counted: {len(boxes)}", fontsize=13)
    plt.show()
    print(f"{Path(path).name}: CNN says {answer} (P(group) = {p_group:.2f}); detector counted {len(boxes)} person(s)")

In [37]:
!ls /content/work/weights /content/work/test_metrics.csv

/content/work/test_metrics.csv

/content/work/weights:
CustomCNN_Aug.weights.h5	   MobileNetV2_scratch.weights.h5
CustomCNN_noAug.weights.h5	   ResNet50_pretrained.weights.h5
MobileNetV2_pretrained.weights.h5  ResNet50_scratch.weights.h5


In [ ]:
# Choose your own photo(s) - you can select several files at once
try:
    from google.colab import files
    uploaded = files.upload()
    for file_name in uploaded:
        predict_photo(file_name)
except ImportError:
    # not in Colab (e.g. Jupyter on your PC): put your photos in a folder called "my_photos"
    for p in sorted(Path("my_photos").glob("*")):
        if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
            predict_photo(p)

## Step 11 – Conclusions (write in your own words)
1. **Best model:** *(see Step 6 – which model has the highest F1 and accuracy?)*
2. **Data augmentation:** *(see Step 7 – did it help the custom CNN? Look at the gap between train and validation curves.)*
3. **Transfer learning:** *(see Step 7 – how much better are the pretrained ResNet50 and MobileNetV2 than the same networks trained from scratch?)*
4. **Efficiency:** *(see Step 8 – is the best model also small and fast? Compare ResNet50 with MobileNetV2.)*
5. **Errors:** *(see Step 9 – which images are hard and why?)*
6. **Limitations and future work:** labels come from automatic rules, only 160 × 160 pixels, only 2 classes (no counting inside the classifier); next steps: larger images, more data, a model that predicts the number of people directly.